In [25]:
import cv2
import numpy as np
import os
N = 10
M = 10
# Путь к директории с изображениями
image_dir = ''  # Замените на путь к вашим изображениям

# Список шаблонов
badger_templates = []
chimpunk_templates = []

# Функция для создания маски
def create_mask(image):
    # Преобразование в оттенки серого
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Бинаризация изображения
    _, mask = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
    return mask

# Загрузка изображений барсуков
for i in range(1, N + 1):  # Замените N на количество изображений барсуков
    img_path = os.path.join(image_dir, f'badger{i}.jpg')
    img = cv2.imread(img_path)
    if img is not None:
        mask = create_mask(img)
        badger_templates.append((img, mask))

# Загрузка изображений бурундуков
for i in range(1, M + 1):  # Замените M на количество изображений бурундуков
    img_path = os.path.join(image_dir, f'chimpunk{i}.jpg')
    img = cv2.imread(img_path)
    if img is not None:
        mask = create_mask(img)
        chimpunk_templates.append((img, mask))

# Загрузка изображения для классификации
test_image_path = "exampleb3.jpg"
test_img = cv2.imread(test_image_path)

# Проверка на пустое изображение
if test_img is None:
    print(f"Ошибка: Не удалось загрузить тестовое изображение {test_image_path}")
    exit(1)  # Завершение программы с кодом ошибки

# Функция для изменения размера шаблона
def resize_template(template, max_width, max_height):
    h, w = template.shape[:2]
    if w > max_width or h > max_height:
        scale = min(max_width / w, max_height / h)
        new_size = (int(w * scale), int(h * scale))
        return cv2.resize(template, new_size)
    return template

# Функция для классификации
def classify_image(test_img):
    best_match = None
    best_value = -1
    best_class = None

    # Создание маски для тестового изображения
    test_mask = create_mask(test_img)

    # Получение размеров тестового изображения
    test_height, test_width = test_img.shape[:2]

    # Сравнение с шаблонами барсуков
    for template, mask in badger_templates:
        resized_template = resize_template(template, test_width, test_height)
        resized_mask = resize_template(mask, test_width, test_height)  # Изменение размера маски
        masked_template = cv2.bitwise_and(resized_template, resized_template, mask=resized_mask)
        result = cv2.matchTemplate(test_img, masked_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(result)
        if max_val > best_value:
            best_value = max_val
            best_match = template
            best_class = "Бурундук"

    # Сравнение с шаблонами бурундуков
    for template, mask in chimpunk_templates:
        resized_template = resize_template(template, test_width, test_height)
        resized_mask = resize_template(mask, test_width, test_height)  # Изменение размера маски
        masked_template = cv2.bitwise_and(resized_template, resized_template, mask=resized_mask)
        result = cv2.matchTemplate(test_img, masked_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(result)
        if max_val > best_value:
            best_value = max_val
            best_match = template
            best_class = "Барсук"

    return best_class, best_value

# Классификация тестового изображения
result_class, confidence = classify_image(test_img)

# Вывод результата
print(f"Классификация: {result_class}, Уверенность: {confidence:.2f}")

Классификация: Барсук, Уверенность: 0.44
